In [1]:
import pandas as pd
import pickle
import numpy as np

In [2]:
with open(r"d:\AML_Detection_Project\bank70_train_graph_with_features.pkl", "rb") as f:
    G = pickle.load(f)

test_df = pd.read_csv(r'd:\AML_Detection_Project\top_bank\test.csv')
print('Number of test cases: {}'.format(len(test_df)))

def rule_based_prediction(row, G):
    from_susp = G.nodes[row['from_node']].get('is_suspicious', 0)
    to_susp   = G.nodes[row['to_node']].get('is_suspicious', 0)
    return int(from_susp or to_susp)

test_df['from_node'] = test_df['from_bank'].astype(str) + '|' + test_df['from_account'].astype(str)
test_df['to_node']   = test_df['to_bank'].astype(str)   + '|' + test_df['to_account'].astype(str)

test_df['rule_pred'] = test_df.apply(lambda r: rule_based_prediction(r, G), axis=1)



Number of test cases: 90551


In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_true = test_df['is_laundering']
y_pred = test_df['rule_pred']

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("📊 Rule-based model performance:")
print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"AUC:       {auc:.3f}")
print("\nConfusion Matrix:\n", cm)
print("\nDetailed classification report:\n", classification_report(y_true, y_pred, zero_division=0))


📊 Rule-based model performance:
Accuracy:  0.979
Precision: 0.002
Recall:    0.025
F1-score:  0.003
AUC:       0.503

Confusion Matrix:
 [[88683  1748]
 [  117     3]]

Detailed classification report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99     90431
           1       0.00      0.03      0.00       120

    accuracy                           0.98     90551
   macro avg       0.50      0.50      0.50     90551
weighted avg       1.00      0.98      0.99     90551



In [6]:
test_df['true_label'] = test_df['is_laundering']
false_pos = test_df[(test_df['rule_pred'] == 1) & (test_df['true_label'] == 0)]
false_neg = test_df[(test_df['rule_pred'] == 0) & (test_df['true_label'] == 1)]

print(f"❌ False Positives: {len(false_pos)}")
print(f"❌ False Negatives: {len(false_neg)}")

❌ False Positives: 1748
❌ False Negatives: 117


In [ ]:
# Find true positives (correct laundering classifications)
true_positives_mask = (y_pred == 1) & (y_true == 1)
true_positive_indices = np.where(true_positives_mask)[0]

print(f"Found {len(true_positive_indices)} correctly classified laundering transactions")
print(f"Row indices (0-indexed): {true_positive_indices}")

Found 3 correctly classified laundering transactions
Row indices (0-indexed): [24826 46398 83496]


: 

In [7]:
def enhanced_rule_based_prediction(row, G, 
                                   out_thresh=10, 
                                   in_thresh=10, 
                                   amount_thresh=None):
    u, v = row['from_node'], row['to_node']
    amt = row.get('amount', 0)

    # Get node attributes safely
    src = G.nodes[u] if u in G.nodes else {}
    dst = G.nodes[v] if v in G.nodes else {}

    # Base flags
    suspicious_src = src.get('is_suspicious', 0)
    suspicious_dst = dst.get('is_suspicious', 0)

    # Graph-based heuristics
    high_out = src.get('weighted_out', 0) > out_thresh
    high_in = dst.get('weighted_in', 0) > in_thresh
    reciprocal = (src.get('reciprocity', 0) > 0.5) or (dst.get('reciprocity', 0) > 0.5)
    high_centrality = (src.get('betweenness', 0) > 0.02) or (dst.get('betweenness', 0) > 0.02)

    # Amount-based rule
    if amount_thresh is not None:
        high_amount = amt > amount_thresh
    else:
        high_amount = False

    # Final rule: OR combination
    return int(
        suspicious_src or suspicious_dst or
        high_out or high_in or
        reciprocal or
        high_centrality or
        high_amount
    )


In [8]:
amount_thresh = test_df['amount'].mean() + 2 * test_df['amount'].std()
test_df['rule_pred_enhanced'] = test_df.apply(lambda r: enhanced_rule_based_prediction(r, G, amount_thresh=amount_thresh), axis=1)

In [10]:
from sklearn.metrics import classification_report, roc_auc_score
print(classification_report(test_df['is_laundering'], test_df['rule_pred_enhanced']))
print("AUC:", roc_auc_score(test_df['is_laundering'], test_df['rule_pred_enhanced']))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     90431
           1       0.00      1.00      0.00       120

    accuracy                           0.00     90551
   macro avg       0.00      0.50      0.00     90551
weighted avg       0.00      0.00      0.00     90551

AUC: 0.5


d:\AML_Detection_Project\aml-function\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\AML_Detection_Project\aml-function\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\AML_Detection_Project\aml-function\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [1]:
from collections import defaultdict

def compute_node_laundering_fractions(G):
    # Initialize counts
    out_counts = defaultdict(int)
    out_laundering = defaultdict(int)
    in_counts = defaultdict(int)
    in_laundering = defaultdict(int)
    
    # Count laundering and total transactions for each node
    for u, v, data in G.edges(data=True):
        out_counts[u] += 1
        in_counts[v] += 1
        if data.get('is_laundering', 0):
            out_laundering[u] += 1
            in_laundering[v] += 1
    
    # Compute fractions
    frac_out_laundering = {node: out_laundering[node]/out_counts[node] if out_counts[node] > 0 else 0 for node in G.nodes}
    frac_in_laundering  = {node: in_laundering[node]/in_counts[node] if in_counts[node] > 0 else 0 for node in G.nodes}
    
    return frac_out_laundering, frac_in_laundering

In [2]:
def rule_based_prediction_fraction(row, frac_out_laundering, frac_in_laundering, threshold=0.5):
    from_node = row['from_node']
    to_node = row['to_node']
    
    from_flag = frac_out_laundering.get(from_node, 0) > threshold
    to_flag = frac_in_laundering.get(to_node, 0) > threshold
    
    return int(from_flag or to_flag)


In [5]:
# Precompute fractions
frac_out, frac_in = compute_node_laundering_fractions(G)

# Predict
test_df['predicted_laundering'] = test_df.apply(
    lambda row: rule_based_prediction_fraction(row, frac_out, frac_in, threshold=0.5),
    axis=1
)


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_true = test_df['is_laundering']
y_pred = test_df['predicted_laundering']

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("📊 Rule-based model performance:")
print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"AUC:       {auc:.3f}")
print("\nConfusion Matrix:\n", cm)
print("\nDetailed classification report:\n", classification_report(y_true, y_pred, zero_division=0))


📊 Rule-based model performance:
Accuracy:  0.979
Precision: 0.002
Recall:    0.025
F1-score:  0.003
AUC:       0.503

Confusion Matrix:
 [[88683  1748]
 [  117     3]]

Detailed classification report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99     90431
           1       0.00      0.03      0.00       120

    accuracy                           0.98     90551
   macro avg       0.50      0.50      0.50     90551
weighted avg       1.00      0.98      0.99     90551

